# Модуль 5. Паттерны: State, Strategy, Command

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 5 — Паттерны: State, Strategy, Command  
**Теория:** 8 ч | **Практика:** 10 ч | **Самостоятельная работа:** 8 ч

---

## О чём этот модуль

В модуле 4 мы научились обновлять мир с фиксированным шагом и выделять компоненты. Но остались три архитектурные проблемы:

1. **Состояния игры смешаны с логикой.** Меню, расследование, диалог, бой — всё это разные режимы, но код обрабатывает их через флаги и `if/elif`.
2. **Поведение Кузи зашито в класс.** Если завтра он должен вести себя агрессивно или трусливо — придётся переписывать класс.
3. **Действия игрока необратимы.** Игрок не может отменить ход или вернуться на шаг назад.

Три паттерна решают эти проблемы:

- **State** — объект меняет поведение при изменении внутреннего состояния. Конечный автомат (FSM) без `if/elif`.
- **Strategy** — взаимозаменяемые алгоритмы. Поведение можно менять на лету, не трогая контекст.
- **Command** — действие как объект. Позволяет ставить в очередь, логировать и отменять.

## Где мы находимся в проекте

После модуля 4 проект «Тайна исчезнувшего дива» имеет:

| Слой | Что есть |
|---|---|
| **Абстракции** | `GameObject` (ABC), протоколы способностей |
| **Сущности** | `Detective`, `Div`, `Location`, `Clue`, `Entity` + компоненты |
| **Системы** | Командные (`InvestigationSystem`, ...) и обновления (`MovementSystem`, ...) |
| **Оркестрация** | `Game`, `GameLoop`, `GameState` |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer` |

**Чего не хватает:** состояния игры не формализованы, поведение Кузи статично, действия нельзя отменить.

## Что вы изучите

1. Конечные автоматы (FSM) и паттерн State.
2. Иерархические состояния и подсостояния.
3. Паттерн Strategy: стратегии поведения.
4. Разница между State и Strategy.
5. Паттерн Command: действия как объекты.
6. Undo/Redo через стек команд.
7. Интеграция трёх паттернов в игровой проект.

## Содержание
1. Конечные автоматы (FSM)
2. Паттерн State
3. Иерархические и параллельные состояния
4. Паттерн Strategy
5. State vs Strategy
6. Паттерн Command
7. Undo/Redo через стек
8. Интеграция в проект
9. Итоги и литература

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# 1. Конечные автоматы (FSM)

## 1.1. Что такое FSM

**Конечный автомат** (Finite State Machine, FSM) — это модель, которая:
- имеет **фиксированный набор состояний**;
- в каждый момент находится **ровно в одном** состоянии;
- переходит между состояниями по **событиям** или **условиям**;
- в каждом состоянии ведёт себя **по-разному**.

**Пример из игры:** персонаж может быть `idle` (стоит), `run` (бежит), `jump` (прыгает), `fall` (падает). В каждый момент — ровно одно из этих состояний.

## 1.2. FSM без паттерна: enum + if/elif

Простейшая реализация — enum состояний и `if/elif` по нему:

```python
from enum import Enum

class State(Enum):
    IDLE = "idle"
    RUN = "run"
    JUMP = "jump"

class Character:
    def __init__(self):
        self.state = State.IDLE

    def update(self, dt):
        if self.state == State.IDLE:
            self.idle_update(dt)
        elif self.state == State.RUN:
            self.run_update(dt)
        elif self.state == State.JUMP:
            self.jump_update(dt)
```

**Проблемы:**
- Каждое новое состояние — новый `elif` (нарушает OCP).
- Логика разных состояний размазана по методам `idle_update`, `run_update`.
- Невозможно добавить общее поведение для группы состояний (например, «в воздухе»).

## 1.3. FSM через паттерн State

**State** заменяет `if/elif` полиморфизмом: каждое состояние — отдельный класс с методами `enter()`, `update()`, `exit()`.

```python
class State(ABC):
    @abstractmethod
    def enter(self, char): ...
    @abstractmethod
    def update(self, char, dt): ...
    @abstractmethod
    def exit(self, char): ...
```

**Ключевая идея:** персонаж **делегирует** `update()` текущему состоянию. Состояние само решает, когда и куда перейти.

## 1.4. Зачем FSM в играх

- **AI врагов:** патрулирование → погоня → атака → отступление.
- **Состояния игры:** меню → игра → пауза → диалог → победа.
- **Анимации:** idle → walk → run → jump.
- **Интерфейсы:** главное меню → настройки → игра.

## 1.5. Как это улучшит проект

В «Тайне исчезнувшего дива» FSM нужен для:
- **состояний игры** (`MenuState`, `InvestigationState`, `DialogueState`, `VictoryState`);
- **состояний Кузи** (`IdleForm`, `ScoutForm`, `CombatForm`);
- **состояний детектива** (`Idle`, `Investigating`, `Resting`).

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [State](https://gameprogrammingpatterns.com/state.html) — каноническое объяснение с примером Heroine.
- [python-statemachine](https://pypi.org/project/python-statemachine/) — популярная библиотека FSM для Python.
- [Refactoring Guru: State](https://refactoring.guru/design-patterns/state)

---
# 2. Паттерн State

## 2.1. Структура паттерна

| Роль | Что делает | Пример |
|---|---|---|
| **Context** | Хранит текущее состояние, делегирует ему поведение | `Character` |
| **State** | Абстрактный интерфейс состояний | `CharacterState` |
| **ConcreteState** | Конкретные состояния | `IdleState`, `RunState`, `JumpState` |

## 2.2. Жизненный цикл состояния

Каждое состояние имеет три метода:

| Метод | Когда вызывается | Что делает |
|---|---|---|
| `enter(context)` | При входе в состояние | Инициализация, запуск анимации |
| `update(context, dt)` | Каждый кадр | Логика состояния, проверка перехода |
| `exit(context)` | При выходе из состояния | Очистка, остановка анимации |

**Зачем `enter`/`exit`:** асимметричные side effects. Например, при входе в `HurtState` персонаж становится неуязвимым, при выходе — уязвимым. Без этих методов пришлось бы дублировать код.

## 2.3. Кто решает о переходе

Есть два подхода:

**1. Состояние решает само** (проще, но состояния знают друг о друге):
```python
class IdleState(State):
    def update(self, char, dt):
        if char.input_x != 0:
            char.state = RunState()  # прямое переключение
```

**2. Контекст решает** (сложнее, но состояния не знают друг о друге):
```python
class Character:
    def update(self, dt):
        next_state = self.state.update(self, dt)
        if next_state:
            self.change_state(next_state)
```

**Рекомендация:** для небольших FSM — первый подход. Для больших — второй (через возврат следующего состояния или через `StateMachine`).

## 2.4. Иерархические состояния

Если несколько состояний имеют общее поведение, можно выделить **родительское состояние**:

```
CharacterState
├── GroundedState
│   ├── IdleState
│   └── RunState
└── AirborneState
    ├── JumpState
    └── FallState
```

`JumpState` и `FallState` наследуются от `AirborneState`, который реализует общую физику (гравитацию). Это уменьшает дублирование.

## 2.5. Как это улучшит проект

**До:** `Game.run()` с флагами `if self.state == 'menu'`, `elif self.state == 'game'`.

**После:** каждое состояние — отдельный класс. `Game` просто делегирует `update` текущему состоянию. Новое состояние = новый класс, без правки `Game`.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [State](https://gameprogrammingpatterns.com/state.html).
- [GameProgrammingPatterns-CN: State](https://github.com/Seeking13/GameProgrammingPatterns-CN/blob/master/02.1-Command.md) — перевод с примерами.
- [Refactoring Guru: State](https://refactoring.guru/design-patterns/state).

---
# 3. Иерархические и параллельные состояния

## 3.1. Иерархические состояния

**Иерархическое состояние** — это состояние, которое содержит подсостояния. Например, `AirborneState` может содержать `JumpState` и `FallState`. Пока персонаж в воздухе, общая логика (гравитация) работает для обоих подсостояний.

**Зачем:** уменьшает дублирование. Вместо того чтобы писать гравитацию в `JumpState` и `FallState`, пишем один раз в `AirborneState`.

## 3.2. Параллельные состояния

**Параллельные состояния** — это независимые FSM, работающие одновременно. Например, персонаж может быть в состоянии `Idle` (движение) и одновременно в состоянии `Poisoned` (эффект отравления).

**Как реализовать:** два отдельных FSM, каждый со своим `update`.

## 3.3. Подводные камни
#### Слишком много состояний
Если FSM разрастается до 20+ состояний, возможно, стоит разбить его на несколько независимых FSM (параллельных) или использовать более сложные модели (behavior trees).

## 3.4. Как это улучшит проект

В «Тайне исчезнувшего дива» иерархия состояний игры может выглядеть так:

```
GameState
├── MenuState
├── PlayingState
│   ├── InvestigatingState
│   ├── DialogueState
│   └── RestingState
└── VictoryState
```

`PlayingState` содержит общую логику (энергия, локации), подсостояния — специфику.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Hierarchical State Machine](https://gameprogrammingpatterns.com/state.html#hierarchical-state-machine).
- [Statecharts](https://statecharts.dev/) — формальная модель иерархических состояний.

---
# 4. Паттерн Strategy

## 4.1. Что такое Strategy

**Strategy** — паттерн, который определяет семейство алгоритмов, инкапсулирует каждый из них и делает их взаимозаменяемыми. Клиент (контекст) может менять стратегию **во время выполнения**.

**Ключевое отличие от State:** в State переход между состояниями обычно **внутренний** (объект сам решает). В Strategy стратегию задаёт **клиент** извне, и она редко меняется сама.

## 4.2. Структура

| Роль | Что делает | Пример |
|---|---|---|
| **Context** | Использует стратегию | `Div` (Кузя) |
| **Strategy** | Интерфейс алгоритма | `BehaviorStrategy` |
| **ConcreteStrategy** | Конкретные алгоритмы | `CautiousStrategy`, `AggressiveStrategy` |

## 4.3. Пример из игры: стратегии поведения Кузи

Кузя может вести себя по-разному:
- **Осторожно:** избегает опасностей, чаще прячется.
- **Агрессивно:** атакует врагов, меньше защищается.
- **Игриво:** случайные действия, больше диалогов.

Каждая стратегия — отдельный класс с методом `decide(div, state)`.

## 4.4. Strategy vs State

| Критерий | State | Strategy |
|---|---|---|
| Кто меняет | Объект сам (внутренние переходы) | Клиент (внешнее назначение) |
| Цель | Управление состояниями | Выбор алгоритма |
| Пример | FSM персонажа | Стратегия AI, алгоритм сортировки |

**Правило:** если переходы — часть логики объекта, это State. Если алгоритм — сменная деталь, это Strategy.

## 4.5. Как это улучшит проект

**До:** поведение Кузи зашито в `Div.scout()`. Чтобы изменить — правим класс.

**После:** `Div` содержит ссылку на `BehaviorStrategy`. Можно менять стратегию на лету: после события «Кузя испугался» — поставить `CautiousStrategy`.

## Для углублённого изучения
- Nystrom R. «Game Programming Patterns»: [Strategy](https://gameprogrammingpatterns.com/strategy.html).
- [Refactoring Guru: Strategy](https://refactoring.guru/design-patterns/strategy).
- [Stack Overflow: Strategy vs State](https://stackoverflow.com/questions/1658192/what-is-the-difference-between-strategy-design-pattern-and-state-design-pattern).

---
# 5. Паттерн Command

## 5.1. Что такое Command

**Command** — паттерн, который инкапсулирует запрос как объект. Это позволяет:
- параметризовать клиентов разными запросами;
- ставить запросы в очередь;
- логировать запросы;
- поддерживать отмену (undo) и повтор (redo).

**Ключевая идея:** «Команда — это реифицированный вызов метода» (Nystrom). Вместо `obj.method(args)` создаём объект `Command`, который знает, что и как вызвать.

## 5.2. Структура
| Роль | Что делает | Пример |
|---|---|---|
| **Command** | Интерфейс с `execute()` и (опционально) `undo()` | `ActionCommand` |
| **ConcreteCommand** | Конкретная команда | `MoveCommand`, `InvestigateCommand` |
| **Invoker** | Вызывает команды | `Game` |
| **Receiver** | Объект, над которым выполняется действие | `Detective`, `GameState` |

## 5.3. Undo/Redo через стек

**Стек команд:**
- При выполнении команды — `push` в стек.
- При отмене — `pop` и вызов `undo()`.
- При повторе — `push` обратно и `execute()`.

```python
class CommandHistory:
    def __init__(self):
        self._history = []
    def execute(self, cmd):
        cmd.execute()
        self._history.append(cmd)
    def undo(self):
        if self._history:
            self._history.pop().undo()
```

## 5.4. Как это улучшит проект

**До:** действия игрока выполняются сразу и необратимо.

**После:** каждое действие — команда. Можно отменить последний ход, посмотреть историю, сохранить игру как список команд.

## Для углублённого изучения
- Nystrom R. «Game Programming Patterns»: [Command](https://gameprogrammingpatterns.com/command.html).
- [patternsgameprog.com: Command pattern](https://patternsgameprog.com/discover-python-and-patterns-12-command-pattern).
- [Refactoring Guru: Command](https://refactoring.guru/design-patterns/command).

---
# 6. Интеграция State, Strategy и Command

## 6.1. Как паттерны сочетаются

В игровом проекте три паттерна часто работают вместе:
- **State** управляет режимами игры (`Menu`, `Playing`, `Victory`).
- **Strategy** управляет поведением AI (Кузя, враги).
- **Command** инкапсулирует действия игрока (ход, осмотр, отмена).

## 6.2. Пример взаимодействия

```
Game (Invoker)
├── current_state: GameState (State)
│   ├── MenuState
│   ├── PlayingState
│   └── VictoryState
├── command_history: CommandHistory
│   ├── MoveCommand
│   └── InvestigateCommand
└── div: Div
    └── strategy: BehaviorStrategy (Strategy)
```

## 6.3. Что появится в проекте

| Паттерн | Где | Что улучшит |
|---|---|---|
| **State** | `GameState` | Меню, игра, победа — без `if/elif` |
| **Strategy** | `Div` | Поведение Кузи можно менять |
| **Command** | `CommandHistory` | Отмена действий, история |

## 6.4. Как это подготовит следующие модули

- **Модуль 6 (Observer, Factory, Singleton):** команды будут генерировать события, состояния — подписываться на них.
- **Модуль 7 (ECS):** состояния и стратегии станут компонентами.
- **Финальный проект:** три паттерна — основа архитектуры.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [State](https://gameprogrammingpatterns.com/state.html), [Strategy](https://gameprogrammingpatterns.com/strategy.html), [Command](https://gameprogrammingpatterns.com/command.html).
- Gamma E. et al. «Design Patterns» (GoF), главы про State, Strategy, Command.

**Углублённая:**
- [Refactoring Guru: State](https://refactoring.guru/design-patterns/state), [Strategy](https://refactoring.guru/design-patterns/strategy), [Command](https://refactoring.guru/design-patterns/command).
- [patternsgameprog.com: Command pattern](https://patternsgameprog.com/discover-python-and-patterns-12-command-pattern).
- [python-statemachine](https://pypi.org/project/python-statemachine/) — библиотека FSM.

## Что дальше

В практической работе вы добавите в «Тайну исчезнувшего дива»:
- состояния игры (меню, расследование, победа);
- стратегии поведения Кузи;
- команды с отменой.